# 05. Model Architecture & Training

This notebook defines, tests, and trains the Sequence-to-Sequence (Seq2Seq) LSTM model:
1. **Encoder**: Embedding (256) + LSTM (512, 1 layer, dropout=0.2).
2. **Decoder**: Embedding (256) + LSTM (512) + Linear projection (`OUTPUT_DIM`).
3. **Seq2Seq**: Ties Encoder and Decoder with teacher forcing (`ratio = 0.5`).
4. **Parameter Counting**: Trainable parameter breakdown (~78.4M parameters).
5. **Loss & Optimizer**: `CrossEntropyLoss(ignore_index=0)` and `Adam(lr=0.001)`.
6. **Sanity Training**: Executes a 100-batch sanity run and verifies loss convergence.
7. **Checkpointing**: Saves checkpoints with architecture configs to `MODEL_DIR`.


In [ ]:
# Environment & Path Setup
# If running on Google Colab, uncomment the lines below:
# from google.colab import drive
# drive.mount('/content/drive')
# %cd /content/english-amharic-nmt

import os
import sys
from pathlib import Path

# Add project root to sys.path
PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.utils.paths import get_data_paths
from src.utils.seed import set_seed

# Configure data directory (can be overridden by DATA_ROOT environment variable)
# On Colab: DATA_ROOT = "/content/drive/MyDrive/english-amharic-nmt-data"
DATA_ROOT = os.getenv("DATA_ROOT", str(PROJECT_ROOT / "data"))
paths = get_data_paths(DATA_ROOT)
paths.ensure_directories()
set_seed(42)

print("Project root:", PROJECT_ROOT)
print("Data directory:", paths.data_root)


## 1. Hardware Environment Check

In [ ]:
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("PyTorch Version:", torch.__version__)
print("Device selected:", device)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("CUDA Version:", torch.version.cuda)


## 2. Load Data & Vocabularies

In [ ]:
import pandas as pd
from src.data.vocabulary import load_vocab, PAD_IDX
from src.data.dataset import TranslationDataset, get_dataloader

eng_vocab = load_vocab(paths.eng_vocab_path)
amh_vocab = load_vocab(paths.amh_vocab_path)

train_df = pd.read_csv(paths.train_filtered_path)
train_dataset = TranslationDataset(train_df, eng_vocab, amh_vocab, max_len=70)
train_loader = get_dataloader(train_dataset, batch_size=64, shuffle=True)

INPUT_DIM = len(eng_vocab)
OUTPUT_DIM = len(amh_vocab)

print(f"Encoder Input Dim:  {INPUT_DIM:,}")
print(f"Decoder Output Dim: {OUTPUT_DIM:,}")


## 3. Instantiate Encoder, Decoder, and Seq2Seq Model

In [ ]:
from src.models.encoder import Encoder
from src.models.decoder import Decoder
from src.models.seq2seq import Seq2Seq, count_parameters

EMBEDDING_DIM = 256
HIDDEN_DIM = 512
NUM_LAYERS = 1
DROPOUT = 0.2

encoder = Encoder(
    input_dim=INPUT_DIM,
    embedding_dim=EMBEDDING_DIM,
    hidden_dim=HIDDEN_DIM,
    num_layers=NUM_LAYERS,
    dropout=DROPOUT,
    pad_idx=PAD_IDX,
)

decoder = Decoder(
    output_dim=OUTPUT_DIM,
    embedding_dim=EMBEDDING_DIM,
    hidden_dim=HIDDEN_DIM,
    num_layers=NUM_LAYERS,
    dropout=DROPOUT,
    pad_idx=PAD_IDX,
)

model = Seq2Seq(encoder, decoder, device=device).to(device)

total_params = count_parameters(model)
print(f"Total Trainable Parameters: {total_params:,} ({total_params/1e6:.2f} Million)")
print(f" - Encoder parameters: {count_parameters(encoder):,}")
print(f" - Decoder parameters: {count_parameters(decoder):,}")
print(f" - Output FC layer:    {count_parameters(decoder.fc_out):,}")


## 4. Test Forward Pass on Single Batch
We verify the tensor shapes through Encoder and Decoder:
- Inputs transposed to `[seq_len, batch_size]` for the LSTM.
- Output tensor shape matches `[trg_len, batch_size, output_dim]`.


In [ ]:
src_batch, trg_batch = next(iter(train_loader))
src_batch = src_batch.to(device).transpose(0, 1)
trg_batch = trg_batch.to(device).transpose(0, 1)

with torch.no_grad():
    output = model(src_batch, trg_batch, teacher_forcing_ratio=0.5)

print("Source shape: ", src_batch.shape)
print("Target shape: ", trg_batch.shape)
print("Output shape: ", output.shape)
assert output.shape == (70, 64, OUTPUT_DIM), f"Unexpected output shape: {output.shape}"


## 5. Loss Criterion and Optimizer Setup

In [ ]:
import torch.nn as nn
import torch.optim as optim

criterion = nn.CrossEntropyLoss(ignore_index=PAD_IDX)
optimizer = optim.Adam(model.parameters(), lr=0.001)

print("Loss function:", criterion)
print("Optimizer:    ", optimizer)


## 6. Sanity Training Run (100 Batches)
We execute a 100-batch training sanity check to ensure gradients propagate and the loss decreases monotonically as expected.

In [ ]:
from src.training.train import run_sanity_training

losses, elapsed = run_sanity_training(
    model=model,
    loader=train_loader,
    optimizer=optimizer,
    criterion=criterion,
    num_batches=100,
    clip=1.0,
    teacher_forcing_ratio=0.5,
    device=device,
    print_interval=10
)

print(f"\nSanity run completed in {elapsed:.2f} seconds ({elapsed/60:.2f} minutes)")
print(f"First Batch Loss: {losses[0]:.4f}")
print(f"Last Batch Loss:  {losses[-1]:.4f}")


## 7. Save Checkpoints
We save:
1. `seq2seq_sanity_100_batches.pt`: The state after the 100-batch sanity run.
2. `seq2seq_baseline_untrained.pt`: Clean baseline weights initialized from scratch with metadata.


In [ ]:
from src.training.checkpoint import save_checkpoint

# 1. Save sanity check checkpoint
sanity_path = paths.models_dir / "seq2seq_sanity_100_batches.pt"
save_checkpoint({
    "model_state_dict": model.state_dict(),
    "optimizer_state_dict": optimizer.state_dict(),
    "loss": losses[-1],
    "batch": 100,
    "config": {
        "embedding_dim": EMBEDDING_DIM,
        "hidden_dim": HIDDEN_DIM,
        "num_layers": NUM_LAYERS,
        "dropout": DROPOUT,
        "batch_size": 64,
        "max_len": 70,
        "learning_rate": 0.001,
        "teacher_forcing_ratio": 0.5,
    }
}, sanity_path)
print("Saved sanity checkpoint to:", sanity_path)

# 2. Re-initialize fresh baseline model and save
fresh_encoder = Encoder(INPUT_DIM, EMBEDDING_DIM, HIDDEN_DIM, NUM_LAYERS, DROPOUT, pad_idx=PAD_IDX)
fresh_decoder = Decoder(OUTPUT_DIM, EMBEDDING_DIM, HIDDEN_DIM, NUM_LAYERS, DROPOUT, pad_idx=PAD_IDX)
fresh_model = Seq2Seq(fresh_encoder, fresh_decoder, device=device).to(device)
fresh_optimizer = optim.Adam(fresh_model.parameters(), lr=0.001)

untrained_path = paths.models_dir / "seq2seq_baseline_untrained.pt"
save_checkpoint({
    "model_state_dict": fresh_model.state_dict(),
    "optimizer_state_dict": fresh_optimizer.state_dict(),
    "config": {
        "embedding_dim": EMBEDDING_DIM,
        "hidden_dim": HIDDEN_DIM,
        "num_layers": NUM_LAYERS,
        "dropout": DROPOUT,
        "batch_size": 64,
        "max_len": 70,
        "learning_rate": 0.001,
        "teacher_forcing_ratio": 0.5,
        "parameters": count_parameters(fresh_model)
    }
}, untrained_path)
print("Saved untrained baseline checkpoint to:", untrained_path)
